> Additional pin issue

In [ ]:
#| default_exp failure_investigation.additional_pin

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import tensorflow as tf
from fastcore.all import *
import numpy as np
import os
import sys
import yaml
import cv2
from fastcore.imports import *
from PIL import Image

In [ ]:
os.getenv('PATH')

In [ ]:
custom_lib_path = Path("/home/ai_warstein/homes/goni/custom_libs")
sys.path.append(str(custom_lib_path))

In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path="/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/private_easy_pin_detection/.env")

In [ ]:
CV_TOOLS = os.getenv("CV_TOOLS")
P_EPD= os.getenv("PRIVATE_EASY_PIN_DETECTION")
sys.path.append(CV_TOOLS)
sys.path.append(P_EPD)

In [ ]:
from private_easy_pin_detection.three_channel_dataset import *
from private_easy_pin_detection.mask_generation import *
from private_easy_pin_detection.mask_generation_two_image import *
from private_easy_pin_detection.core import *

In [ ]:
from cv_tools.core import *

In [ ]:
import matplotlib as mpl
mpl.rcParams['image.cmap'] = 'gray'

# Check for rotation with reference image

In [ ]:
im_path = Path(r'/home/ai_sintercra.work/Fail_investigation/additional/FailSaved20240709/FailSaved20240709_unzip/main_im1')
ref_im_path = Path(r'/home/ai_easypid.work/Reference_images')
ref_im_path_2b = Path(ref_im_path, 'ref_image_2B.png')
rf_2b = read_img(ref_im_path_2b)
images_ = im_path.ls()
img_ = read_img(images_[1])

In [ ]:
ref_fn_frm_fn(images_[0])

In [ ]:
for i in images_:
    mod_name = get_m_name(Path(f'{i}').name)
    pin_type = get_pin_type()[mod_name]
    try:
        if '2B' in pin_type:
            ref_name = 'ref_image_2B.png'
        elif '1B' in pin_type:
            ref_name = 'ref_image_1B.png'
        elif 'SMART' in pin_type:
            ref_name = 'ref_image_SMART.png'
    except Exception as e:
        print(f'Error: {e}')
        continue

In [ ]:
pin_map()

In [ ]:
show_(img_)

In [ ]:
find_rotation_angle(rf_2b, img_)

# Idea:

1. prepare image 
2. prediciton with current single image model
3. prediction with current double image model
4. Try to solve doubple image addtional pin problem with erosion and dilation 
5. Try to see prediction with smaller patch model

 ### 1. Prepare image both single and double image model

In [ ]:
data_path = Path("/home/ai_sintercra.work/Fail_investigation/Missing_lead/FailTest_copy/FailTest_copy_unzip")
data_path.ls()
im1_path=Path(data_path, 'main_im1_cropped')
im2_path=Path(data_path, 'main_im2_cropped')
pr_im_path=Path(data_path, 'main_pr_cropped')
config_path=Path(P_EPD, 'private_easy_pin_detection')
on_c_config_path=Path(config_path, 'config_tst.yaml')
two_i_config_path=Path(config_path, 'config_test_two_image.yml')

In [ ]:
def read_config(config_path:str):
    with open(f'{config_path}', 'r') as f:
        config = yaml.safe_load(f)
    return config

In [ ]:
config_data=read_config(on_c_config_path)
two_i_config_data=read_config(two_i_config_path)

In [ ]:
mdl = load_model(config_data)

In [ ]:
rot_path=Path(r'/home/ai_sintercra.work/Fail_investigation/rotation')

In [ ]:
im_list =[]
img = Image.open(rot_path.ls()[0])
for i in range(2):
    img.seek(i)
    im_list.append(img.copy())


In [ ]:
img_2 = np.array(im_list[1])
img_1 = np.array(im_list[0] )

In [ ]:
show_(img_1)

In [ ]:
show_(img_2)

In [ ]:
img1_f = img_1.astype(np.float32)
img1_f_n = img1_f/255.0
img_1_f_n_b = (img1_f_n >=0.5).astype(np.float32)
img_1_f_n_b_t = img_1_f_n_b[None, ...,None]

In [ ]:
def process_img(img):
    if not isinstance(img, np.ndarray):
        img = np.array(img)
    img = img.astype(np.float32)
    img = img/255.0 
    img = img[None, ...,None]
    return img

In [ ]:
def img1_threshold(img1, threshold=0.3):
    img1 = img1.astype(np.float32)
    img1 = (img1 > threshold).astype(np.float32)
    return img1

In [ ]:
def erosion_(img, kernel_size=5, iterations=1):
    x = img
    for i in range(iterations):
        x = tf.keras.layers.MaxPooling2D(pool_size=kernel_size, strides=[1,1], padding='same')(x)
    return x

In [ ]:
img_2_p = process_img(img_2)
img_1_p = process_img(img_1)

# First do threhsoling of image 1 

In [ ]:
msk = mdl.predict(img_2_p)
msk.shape

In [ ]:
show_(msk[0,...,0] > 0.5) 

## Solution would be 

1. First threshold final mask
2. multiply with image1 threshold
3. reverse
4. erosion 2 time
5. reverse ersone 5 times
6. multiply step1 mask * step 5 result 


In [ ]:
mdl2_i = load_model(two_i_config_data)

In [ ]:
def predict_msk_two_image(
    model:tf.keras.Model,
    im1_path:str, # file path im1 without file name
    im2_fn:str,# file name im2
    config:dict,
    save_path:Path=None,
    ):

    name_ = Path(im2_fn).name
    im1_name = name_.replace('image2', 'image1')
    print(im1_name)
    im1_fn = Path(im1_path, im1_name)
    
    # read images
    im1_img = read_img(im1_fn)
    im2_img = read_img(im2_fn)
    stack_image = process_image_im1_im2_j(
                                        im1=im1_img,
                                        im2=im2_img,
    )
    mdl_inp = tf.transpose(
        stack_image, 
        perm=(0,3,1,2)
        )
    predictions = model.predict(mdl_inp)
    pred = pred_to_np255int8(predictions)
    return pred

In [ ]:
stack_image = process_image_im1_im2_j(
    im1=img_1, 
    im2=img_2)
mdl_inp = tf.transpose(
        stack_image, 
        perm=(0,3,1,2)
        )

In [ ]:

f_out = mdl2_i.predict(mdl_inp)

In [ ]:
show_(f_out[0,...,0])

In [ ]:
im2_path.ls()[0]

In [ ]:
msk = predict_msk(mdl, config=config_data,im_file=f'{im2_path.ls()[0]}')

In [ ]:
msk_two_i = predict_msk_two_image(
    mdl2_i, 
    im1_path=im1_path, 
    im2_fn=f'{im2_path.ls()[1]}', config=two_i_config_data)

In [ ]:
show_(msk_two_i)

In [ ]:
cntrs=find_contours_binary(msk)
cntrs_t=find_contours_binary(msk)

In [ ]:

ars_ = [cv2.contourArea(i) for i in cntrs_t]

In [ ]:
rot_path = Path('/home/ai_sintercra.work/Fail_investigation/rotation')
img = Image.open(rot_path.ls()[0])

In [ ]:
show_(msk)

In [ ]:
config_path=

In [ ]:
path = Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im1_cropped')
path.ls()

In [ ]:
fn = path.ls()[0]

In [ ]:
im = np.expand_dims((read_img(fn)/255.0).astype(np.float32),-1)[None,...]
im.shape

In [ ]:
mp_img = tf.keras.layers.MaxPool2D(pool_size=(2,2), padding='same', strides=2)(im)

In [ ]:
mp_img[0].shape

In [ ]:
show_(mp_img[0])

In [ ]:
show_(mp_img[0])

In [ ]:

class RotatedRectangleFitter(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(RotatedRectangleFitter, self).__init__(**kwargs)

    def call(self, binary_masks):
        def fit_rotated_rectangles(mask):
            # Convert the binary mask to a NumPy array
            mask = mask.numpy()

            # Find the contours in the binary mask
            contours = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]

            rotated_rectangles = []
            for contour in contours:
                # Fit a rotated rectangle to the contour
                rect = cv2.minAreaRect(contour)
                rotated_rectangles.append(rect)

            # Convert the rotated rectangles to a TensorFlow tensor
            rotated_rectangles = tf.convert_to_tensor(rotated_rectangles, dtype=tf.float32)

            return rotated_rectangles

        # Apply the fit_rotated_rectangles function to each binary mask in the batch
        rotated_rectangles = tf.map_fn(fit_rotated_rectangles, binary_masks, fn_output_signature=tf.float32)

        return rotated_rectangles

In [ ]:
M:\Fail_start20240402\_unzip\main_im2_cropped_masks\two_image_modelthrsh5thrsh1.keras\failed\additional\masks

In [ ]:
b_m_path = Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped_masks/two_image_modelthrsh5thrsh1.keras/failed/additional/masks')

In [ ]:
b_m_path.ls()

In [ ]:
msk = read_img(b_m_path.ls()[0])

In [ ]:
def prp_mdl(
            img):
    return np.expand_dims((img/255.0).astype(np.float32),axis=-1)[None,...]

In [ ]:
p_msk = prp_mdl(msk)

In [ ]:
import cv2

In [ ]:
binary_masks = tf.random.uniform([4, 64, 64, 1], minval=0, maxval=2, dtype=tf.int32)

In [ ]:
rotated_rectangle_fitter = RotatedRectangleFitter()

# Apply the layer to the binary masks
rotated_rectangles = rotated_rectangle_fitter(binary_masks)

In [ ]:
import cv2
import numpy as np

def detect_circle(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (9, 9), 2)
    circles = cv2.HoughCircles(blurred, cv2.HOUGH_GRADIENT, dp=1, minDist=50,
                               param1=50, param2=30, minRadius=10, maxRadius=100)
    
    if circles is not None:
        circles = np.round(circles[0, :]).astype("int")
        x, y, r = circles[0]
        return (x, y)
    return None

def calculate_angle(point1, point2):
    dx = point2[0] - point1[0]
    dy = point2[1] - point1[1]
    return np.degrees(np.arctan2(dy, dx))

def rotate_image(image, angle):
    center = tuple(np.array(image.shape[1::-1]) / 2)
    rot_mat = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(image, rot_mat, image.shape[1::-1], flags=cv2.INTER_LINEAR)

# Load reference and test images
reference_image = cv2.imread('reference_image.jpg')
test_image = cv2.imread('test_image.jpg')

# Detect circles in both images
ref_center = detect_circle(reference_image)
test_center = detect_circle(test_image)

if ref_center is None or test_center is None:
    print("Could not detect circles in one or both images.")
else:
    # Calculate the angle difference
    ref_angle = calculate_angle(ref_center, (ref_center[0], 0))
    test_angle = calculate_angle(test_center, (test_center[0], 0))
    rotation_angle = ref_angle - test_angle

    # Rotate the test image
    rotated_test_image = rotate_image(test_image, rotation_angle)

    # Display results
    cv2.imshow('Reference Image', reference_image)
    cv2.imshow('Original Test Image', test_image)
    cv2.imshow('Rotated Test Image', rotated_test_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    # Save the rotated test image
    cv2.imwrite('rotated_test_image.jpg', rotated_test_image)